# 05 – Size-based AP and real-image timing
Extra analysis for the paper and thesis (no retraining):
1. **AP by object size** (COCO small / medium / large) → paper Sect. 4.2 and 4.4
2. **Inference time on real test images** → paper Table 5 and Fig. 4 (550 randomly sampled RDD2022 test images, seed 0; first 10 are warm-up, 540 timed)

Runtime: **GPU → A100** (same as the other notebooks). Edit only the CONFIG cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.171 pycocotools

## CONFIG – change these paths to yours

In [ ]:
DRIVE = '/content/drive/MyDrive/RDD_fair'

RDD_ZIP      = '/content/drive/MyDrive/RDD_SPLIT.zip'
RDD_TEST_GT  = f'{DRIVE}/results/00_dataset/rdd_test_coco.json'
ALB_GT       = f'{DRIVE}/results/00_dataset/albania_coco.json'
ALB_IMG      = '/content/drive/MyDrive/albania_test/test'

YOLO_CKPT    = f'{DRIVE}/results/yolov8m/seed0/checkpoints/best.pt'
YOLO_CKPT_UL = f'{DRIVE}/results/yolov8m/seed0/ultralytics/weights/best.pt'   # fallback
SSD_CKPT     = f'{DRIVE}/results/ssd300_vgg16/seed0/checkpoints/best.pt'
FRCNN_CKPT   = f'{DRIVE}/results/fasterrcnn_r50_fpn_v2/seed0/checkpoints/best.pt'

OUT = f'{DRIVE}/results/size_and_timing'
import os, sys; os.makedirs(OUT, exist_ok=True)
sys.path.append(f'{DRIVE}/code')

In [ ]:
!unzip -q -n "$RDD_ZIP" -d /content/
import glob, json
# index every image by file name, whatever folder it is in
IMG_INDEX = {}
for root in ['/content/RDD_SPLIT', '/content', ALB_IMG]:
    for p in glob.glob(f'{root}/**/*.jp*g', recursive=True) + glob.glob(f'{root}/**/*.png', recursive=True):
        IMG_INDEX.setdefault(os.path.basename(p), p)
print('images indexed:', len(IMG_INDEX))

def img_path(file_name):
    return IMG_INDEX.get(os.path.basename(file_name))

for p in [RDD_TEST_GT, ALB_GT]:
    g = json.load(open(p))
    missing = [im['file_name'] for im in g['images'] if img_path(im['file_name']) is None]
    print(os.path.basename(p), '| images:', len(g['images']), '| boxes:', len(g['annotations']),
          '| MISSING image files:', len(missing), missing[:5])
# expected: RDD test 5,401 / 7,614, Albania 200 / 753, and MISSING = 0 for both
RDD_TEST_IMG = None   # not needed any more; images are found by name

images indexed: 74992
rdd_test_coco.json | images: 5401 | boxes: 7614 | MISSING image files: 0 []
albania_coco.json | images: 200 | boxes: 753 | MISSING image files: 0 []


## Load the three models with your own rdd_fair code (identical to the thesis evaluation)

In [ ]:
import torch, torchvision, numpy as np, cv2
from ultralytics import YOLO
from rdd_fair import tv_models as TVM, config as CFG

try:
    yolo = YOLO(YOLO_CKPT); print('YOLO: loaded checkpoints/best.pt')
except Exception as e:
    print('YOLO: using ultralytics/weights/best.pt'); yolo = YOLO(YOLO_CKPT_UL)
ssd   = TVM.load_for_inference('ssd300_vgg16', SSD_CKPT)
frcnn = TVM.load_for_inference('fasterrcnn_r50_fpn_v2', FRCNN_CKPT)
print('models loaded')

## Part 1 – use the detections fair_eval already saved (results/<model>/seed0/predictions/)

In [ ]:
MODEL_DIRS = {'YOLOv8m': 'yolov8m', 'SSD300-VGG16': 'ssd300_vgg16', 'Faster R-CNN': 'fasterrcnn_r50_fpn_v2'}
SETS = {'RDD2022': RDD_TEST_GT, 'Albania': ALB_GT}
DETS = {}
for mname, mdir in MODEL_DIRS.items():
    pdir = f'{DRIVE}/results/{mdir}/seed0/predictions'
    files = sorted(glob.glob(f'{pdir}/**/*.json', recursive=True))
    print(mname, '->', [os.path.relpath(f, pdir) for f in files])
    test = [f for f in files if 'test' in os.path.basename(f).lower()]
    alb  = [f for f in files if 'alban' in os.path.basename(f).lower()]
    if len(test) == 1: DETS[(mname, 'RDD2022')] = test[0]
    if len(alb) == 1:  DETS[(mname, 'Albania')] = alb[0]
print()
for k in [(m, s) for m in MODEL_DIRS for s in SETS]:
    print(k, '->', os.path.basename(DETS[k]) if k in DETS else 'NOT FOUND - send Claude the file list above')

YOLOv8m -> ['albania.json', 'test.json', 'val.json']
SSD300-VGG16 -> ['albania.json', 'test.json', 'val.json']
Faster R-CNN -> ['albania.json', 'test.json', 'val.json']

('YOLOv8m', 'RDD2022') -> test.json
('YOLOv8m', 'Albania') -> albania.json
('SSD300-VGG16', 'RDD2022') -> test.json
('SSD300-VGG16', 'Albania') -> albania.json
('Faster R-CNN', 'RDD2022') -> test.json
('Faster R-CNN', 'Albania') -> albania.json


## Part 2 – AP by size

In [ ]:
import numpy as np, pandas as pd, contextlib, io
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
SMALL, LARGE = 32**2, 96**2

def load_gt(path):
    d = json.load(open(path))
    for a in d['annotations']:
        a['area'] = float(a['bbox'][2] * a['bbox'][3]); a.setdefault('iscrowd', 0)
    c = COCO(); c.dataset = d
    with contextlib.redirect_stdout(io.StringIO()): c.createIndex()
    return c

def ap(E, area_idx, cat_idx=None):          # AP@0.5 (IoU index 0), maxDets=100
    p = E.eval['precision'][0, :, :, area_idx, -1]
    cols = [cat_idx] if cat_idx is not None else range(p.shape[1])
    vals = [p[:, k][p[:, k] > -1].mean() for k in cols if (p[:, k] > -1).any()]
    return float(np.mean(vals)) if vals else float('nan')

rows, shares = [], []
for (mname, sname), dpath in sorted(DETS.items(), key=lambda kv: (kv[0][1] != 'RDD2022', kv[0][0])):
    gt = load_gt(SETS[sname])
    with contextlib.redirect_stdout(io.StringIO()):
        dd = json.load(open(dpath)); dd = dd['detections'] if isinstance(dd, dict) and 'detections' in dd else dd
        E = COCOeval(gt, gt.loadRes(dd), 'bbox'); E.params.maxDets = [1, 10, 100]; E.evaluate(); E.accumulate()
    rows.append({'set': sname, 'model': mname, 'mAP@0.5 (all, check)': ap(E, 0),
                 'AP@0.5 small': ap(E, 1), 'AP@0.5 medium': ap(E, 2), 'AP@0.5 large': ap(E, 3)})
    if mname == 'YOLOv8m':   # size shares of the ground truth (same for every model)
        cats = {c['id']: c['name'] for c in gt.dataset['categories']}
        for cid in E.params.catIds:
            A = np.array([a['area'] for a in gt.loadAnns(gt.getAnnIds(catIds=[cid]))])
            if len(A) == 0: continue
            shares.append({'set': sname, 'class': cats[cid], 'boxes': len(A),
                           'small %': 100*(A < SMALL).mean(), 'medium %': 100*((A >= SMALL) & (A <= LARGE)).mean(),
                           'large %': 100*(A > LARGE).mean()})

size_df  = pd.DataFrame(rows).round(3)
share_df = pd.DataFrame(shares).round(1)
size_df.to_csv(f'{OUT}/ap_by_size.csv', index=False); share_df.to_csv(f'{OUT}/gt_size_shares.csv', index=False)
display(size_df); display(share_df)
# CHECK: 'mAP@0.5 (all, check)' must match paper Tables 4 and 7 (0.572, 0.390, 0.578 / 0.247, 0.189, 0.292).
# If not, the class ids or settings differ from fair_eval.py - fix before using the size numbers.

,set,model,"mAP@0.5 (all, check)",AP@0.5 small,AP@0.5 medium,AP@0.5 large
0,RDD2022,Faster R-CNN,0.578,0.445,0.494,0.622
1,RDD2022,SSD300-VGG16,0.390,0.122,0.302,0.469
2,RDD2022,YOLOv8m,0.572,0.419,0.499,0.593
3,Albania,Faster R-CNN,0.292,0.125,0.238,0.296
4,Albania,SSD300-VGG16,0.189,0.009,0.021,0.192
5,Albania,YOLOv8m,0.247,1.000,0.118,0.252


,set,class,boxes,small %,medium %,large %
0,RDD2022,D00,3614,5.3,46.3,48.3
1,RDD2022,D10,1533,4.6,66.2,29.2
2,RDD2022,D20,1524,0.1,15.4,84.5
3,RDD2022,D40,943,22.1,60.4,17.5
4,Albania,D00,273,0.4,1.8,97.8
5,Albania,D10,323,0.0,2.5,97.5
6,Albania,D20,120,0.0,0.0,100.0
7,Albania,D40,37,0.0,2.7,97.3


## Part 3 – inference time on real RDD2022 test images (A100, batch 1, FP32)
Each model gets the raw image as in its own pipeline: YOLO gets the BGR array (Ultralytics letterbox),
the torchvision models get the RGB tensor as in `tv_models.CocoDetDataset` and resize internally.
Timed: preprocessing + forward pass + NMS. Not timed: reading the file from disk.

In [ ]:
import time, random
gt = json.load(open(RDD_TEST_GT)); random.seed(0)
imgs_bgr = [cv2.imread(img_path(im['file_name']), cv2.IMREAD_COLOR) for im in random.sample(gt['images'], 550)]

def yolo_fn(img):
    return yolo.predict(img, imgsz=640, conf=0.001, iou=0.5, max_det=100, verbose=False)

def tv_fn(model):
    @torch.no_grad()
    def f(img):
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        t = torch.from_numpy(rgb).cuda().permute(2, 0, 1).float().div_(255.0)
        return model([t])
    return f

def time_model(fn, n_warm=10):
    for img in imgs_bgr[:n_warm]: fn(img)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    for img in imgs_bgr[n_warm:]: fn(img)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / (len(imgs_bgr) - n_warm) * 1000

print(torch.cuda.get_device_name(0))
t_rows = []
for mname, fn in {'YOLOv8m': yolo_fn, 'SSD300-VGG16': tv_fn(ssd), 'Faster R-CNN': tv_fn(frcnn)}.items():
    ms = time_model(fn)
    t_rows.append({'model': mname, 'ms/image': round(ms, 2), 'FPS': round(1000/ms, 1)})
time_df = pd.DataFrame(t_rows); time_df.to_csv(f'{OUT}/timing_real_images.csv', index=False)
display(time_df)

NVIDIA A100-SXM4-40GB


,model,ms/image,FPS
0,YOLOv8m,12.11,82.6
1,SSD300-VGG16,11.29,88.6
2,Faster R-CNN,26.71,37.4
